In [ ]:
The environment
The environment is a MxN array of integers. Each cell of this environment can have the following values:


 
0 : empty cell
1 : walls obstacle that are non-traversable
2 : mine fields
3 : key  
4 : hostage zone
5 : safe zone
6 : unsafe zone
7 : weak hostile (complex version)
8: strong hostile (complex version)
    
 


All border cells are obstacles. Upon initialization, the environment has:

(N+M)/4 obstacles placed randomly in the maze.
N/2 mine cells placed randomly in the cell.

At each timestep:

the agent decides on an action (move up, left, right or down, pick up refugee, put down a hostage)
the action is sent to the environment
the environment sends back observations, rewards and a boolean that indicates whether the environment terminated.
The environment terminates if the agent reaches a safe country, or if the environement reaches a time limit of 2*((M*N)^2) timesteps.


## State representation

In [ ]:
the agent encounters a state, and then takes action according to the state it's in.

The State Space is the set of all possible situations our taxi could inhabit. 
The state should contain useful information the agent needs to make the right action.

The agent starts in a random empty cell, and has to reach the safe zone. 
Both safe country and unsafe is randomly positioned in another empty cell close to the border.


Observations
The agent receives a dictionary of observations that corresponds to the following:
N (rows) * M (columns) * HZ (hostage zone) * 

target: relative coordinates of the exit
proximity: a 3x3 array that encodes for the value of the cells around the agent.


## Actions

6 actions would be possible:

up, left, right or down, pick up hostage, put down a hostage with key

## Rewards

In [ ]:

When acting, an agent receives a reward depending on the cell it ends up on:

if the agent moves towards an obstacle, it gets a reward of -2 and stays at its original position
if the agent moves towards an empty cell, it gets a reward of -1
if the agent is on a mine cell after its action, it receives a reward of -5
at each timestep, the agent receives an additional reward of -1
when the agent reaches a refugee, it receives a reward of N**2

# Library Imports

In [172]:
import numpy as np
import sys
from contextlib import closing
from io import StringIO
from typing import Optional

In [ ]:
import pygame
import constant
from agent import Agent

class MapSquare:
    def __init__(self, attrib_dict):
        # x and y are the true absolute position in the window
        # map_x and map_y are abstract coordinates to represent the agent position and the target in the 2D map matrix
        # the is_blocked_dict represents a source of information (is_blocked_dict) to know where are the walls in a square
        valid_attributes = ("x", "y", "map_x", "map_y", "is_blocked_dict", "id")
        for key in valid_attributes:
            self.__dict__[key] = attrib_dict[key]
    
    # Function used to draw walls in the corners of the map and walls between the map squares
    def draw_walls(self, surface):
        if(self.is_blocked_dict["left"] == '*'):
            pygame.draw.rect(surface, constant.COLOR_DICT['black'], (self.x-1, self.y, constant.WALL_WIDTH, constant.SQUARE_SIZE),
         constant.SQUARE_EMPTY)
        if(self.is_blocked_dict["right"] == '*'):
            pygame.draw.rect(surface, constant.COLOR_DICT['black'], (self.x+constant.SQUARE_SIZE-constant.WALL_WIDTH+1, self.y,
             constant.WALL_WIDTH, constant.SQUARE_SIZE), constant.SQUARE_EMPTY)
        if(self.is_blocked_dict["top"] == '*'):
            pygame.draw.rect(surface, constant.COLOR_DICT['black'], (self.x, self.y-1,
             constant.SQUARE_SIZE, constant.WALL_WIDTH), constant.SQUARE_EMPTY)
        if(self.is_blocked_dict["bottom"] == '*'):
            pygame.draw.rect(surface, constant.COLOR_DICT['black'], (self.x, self.y+constant.SQUARE_SIZE-constant.SQUARE_EMPTY,
             constant.SQUARE_SIZE, constant.WALL_WIDTH), constant.SQUARE_EMPTY)

    # Function used to draw the target map position
    def draw_target(self, surface):
        pygame.draw.rect(surface, constant.COLOR_DICT['pink'], (self.x+1, self.y+1, constant.SQUARE_SIZE-2, constant.SQUARE_SIZE-2),
         constant.SQUARE_FILLED)
    
    # Function used to draw the key map position
    def draw_key(self, surface):
        pygame.draw.rect(surface, constant.COLOR_DICT['black'], (self.x+1, self.y+1, constant.SQUARE_SIZE-2, constant.SQUARE_SIZE-2),
         constant.SQUARE_FILLED)
    
    # Function used to draw the agent map position
    def draw_agent(self, surface):
        pygame.draw.rect(surface, constant.COLOR_DICT['yellow'], (self.x+4, self.y+4, constant.SQUARE_SIZE-8, constant.SQUARE_SIZE-8),
         constant.SQUARE_FILLED)
        pygame.draw.rect(surface, constant.COLOR_DICT['black'], (self.x+4, self.y+4, constant.SQUARE_SIZE-8, constant.SQUARE_SIZE-8),
         constant.SQUARE_EMPTY)

class Labyrinth:
    def __init__(self, map_file, agent):
        # The labyrinth contain a reference to the agent
        self.agent = agent
        self.create_labyrinth(map_file)

    def process_file(self, map_file):
        # Processes map file content and define the map width (number of columns)
        map_file_content = map_file.read().split("\n")
        # Closed opened map file
        map_file.close()
        return map_file_content, [int(i) for i in map_file_content[0].split(" ")][0], [int(i) for i in map_file_content[0].split(" ")][1]

    # Create labyrinth
    def create_labyrinth(self, map_file):   
        
        map_file_content, map_height, map_width = self.process_file(map_file)
        # map_squares: The 2D map matrix is represented by the bidimensional list of map squares
        # square_line: An auxiliar one-dimensional list of squares, representing a line, that will be appended to map_squares
        square_line = []
        self.map_squares = []
        # Relative coordinate for the 2D map matrix
        real_x_coord = 0
        real_y_coord = 0
        # Build the 2D map matrix by creating all map squares and appending them to the map_squares list
        for i in range(2, len(map_file_content), 2):
            real_x_coord = 0
            square_line = []
            for j in range(1, len(map_file_content[1])-1, 2):
                # Creating and appending map squares to the 2D map matrix
                if(map_file_content[i][j] == '0' or map_file_content[i][j] == '2'):
                    square_line.append(self.create_map_square(map_file_content, (i, j, real_y_coord, real_x_coord, map_width, map_height)))
                # Setting up the goal position in the map
                if(map_file_content[i][j] == '2'):
                    # target_x and target_y represents the goal position in the map
                    self.target_x = real_x_coord
                    # Modification to assure that the y axis is increasing (bottom -> up)
                    #self.target_y = map_height - (real_y_coord + 1)
                    self.target_y = real_y_coord
                # Setting up the goal position in the map
                if(map_file_content[i][j] == '3'):
                    # key_x and key_y represents the goal position in the map
                    self.key_x = real_x_coord
                    # Modification to assure that the y axis is increasing (bottom -> up)
                    #self.target_y = map_height - (real_y_coord + 1)
                    self.key_y = real_y_coord    
                real_x_coord += 1
            self.map_squares.append(square_line)
            real_y_coord += 1

    # Create and return a map square
    def create_map_square(self, map_file_content, coordinates):
        map_square_configuration = {
            'x': constant.LABYRINTH_X+(coordinates[3]*constant.SQUARE_SIZE),
            'y': constant.LABYRINTH_Y+(coordinates[2]*constant.SQUARE_SIZE),
            'map_x': coordinates[3],
            # Modification to assure that the y axis is increasing (bottom -> up)
            #'map_y': (coordinates[5]-(coordinates[2]+1)),
            'map_y': coordinates[2],
            'is_blocked_dict': self.calculate_blocked_dict(map_file_content, coordinates),
            'id': (coordinates[3] + (coordinates[4] * coordinates[2]))
        }
        return MapSquare(map_square_configuration)

    # Create a source of information (is_blocked_dict) to know where are the walls in a square
    # my_type: It corresponds to the map square type (0: normal empty block; 2: target/goal position)
    def calculate_blocked_dict(self, map_file_content, coordinates):
        is_blocked_dict = {
        'top': map_file_content[coordinates[0]-1][coordinates[1]],
        'bottom': map_file_content[coordinates[0]+1][coordinates[1]],
        'left': map_file_content[coordinates[0]][coordinates[1]-1],
        'right': map_file_content[coordinates[0]][coordinates[1]+1],
        'my_type': map_file_content[coordinates[0]][coordinates[1]]
        }
        return is_blocked_dict

    # Draw all 
    def draw(self, surface):
        for i in range(len(self.map_squares)):
            for j in range(len(self.map_squares[i])):
                # Draw normal map square
                self.map_squares[i][j].draw_walls(surface)
                # Draw pink background on target map position
                if(self.map_squares[i][j].map_x == self.target_x and self.map_squares[i][j].map_y == self.target_y):
                    self.map_squares[i][j].draw_target(surface)
                # Draw pink background on target map position
                if(self.map_squares[i][j].map_x == self.target_x and self.map_squares[i][j].map_y == self.target_y):
                    self.map_squares[i][j].draw_target(surface)
                # Draw yellow square indicating the agent position
                if(self.map_squares[i][j].map_x == self.agent.map_x and self.map_squares[i][j].map_y == self.agent.map_y):
                    self.map_squares[i][j].draw_agent(surface)

pygame 2.1.2 (SDL 2.0.18, Python 3.8.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import env_config as constant

constant.COLOR_DICT['white']

(255, 255, 255)

In [ ]:
import pygame
import env_config as constant
#from q_learning import QLearning
from agent_class import Agent
from labyrinth_class import Labyrinth


def create_labyrinth(agent):
     # Labyrinth map creation
    file_address = "example_map.txt"
    map_file = open(file_address, "r")
    labyrinth = Labyrinth(map_file, agent)
    return labyrinth

def create_agent():
    # Agent configuration and creation
    agent_config_dict = {
    'map_x': constant.AGENT_MAP_X,
    'map_y': constant.AGENT_MAP_Y,
    'direction': constant.AGENT_DEFAULT_DIRECTION
    }
    agent = Agent(agent_config_dict)
    return agent

def create_window_objects():
    # Font creation
    monospace_font = pygame.font.SysFont(constant.FONT_NAME, constant.FONT_SIZE)

    # Timed event
    game_step_timed_event = constant.GAME_STEP_EVENT_ID
    pygame.time.set_timer(game_step_timed_event, constant.GAME_STEP_DELAY)

    # Defined FPS and FPS clock creation
    fps_clock = pygame.time.Clock()

    return monospace_font, fps_clock, game_step_timed_event

def create_game_window():

    # Initializing pygame and font
    pygame.init()
    pygame.font.init()

    # Set game window title
    pygame.display.set_caption(constant.WINDOW_TITLE)

    # Screen dimensions
    screen_dimensions = constant.WIDTH, constant.HEIGHT

    # Screen creation
    screen = pygame.display.set_mode(screen_dimensions)
    
    return screen

# Draw Q matrix on screen
def draw_matrix_text(font, screen, object_references):
    # object_references[0]: agent
    # object_references[1]: labyrinth
    # object_references[2]: q_learning_solver
    list_of_lines, state_text, action_text = create_matrix_texts(font, screen, object_references)
    below_map_y = object_references[1].map_squares[len(object_references[1].map_squares)-1][0].y+(2.5 * constant.FONT_SIZE)

    screen.blit(state_text, (50, below_map_y+(7*constant.FONT_SIZE)+((len(list_of_lines)/2)*0.8*constant.FONT_SIZE)))
    screen.blit(action_text, (200, below_map_y+(5.8*constant.FONT_SIZE)))
    for i in range(len(list_of_lines)):
        screen.blit(list_of_lines[i], (120, below_map_y+(7*constant.FONT_SIZE)+(i*0.9*constant.FONT_SIZE)))

# Draw the labyrinth map square ids at the top-left corner of a map square
def draw_labyrinth_map_square_id_text(font, screen, object_references):
    for i in range(len(object_references[1].map_squares)):
        for j in range(len(object_references[1].map_squares[i])):
            test_text = font.render(str(object_references[1].map_squares[i][j].id), constant.SMOOTH_EDGES,
                constant.COLOR_DICT['black'])
            screen.blit(test_text, (object_references[1].map_squares[i][j].x, object_references[1].map_squares[i][j].y))

# Draw texts for agent's current position, current epoch, total amount of steps and current exploration_chance from q-learning_solver
def draw_text_on_screen(font, screen, object_references):
    # object_references[0]: agent
    # object_references[1]: labyrinth
    # object_references[2]: q_learning_solver
    
    # Get list of texts to draw on screen
    list_of_texts = create_texts(font, screen, object_references)

    below_map_y = object_references[1].map_squares[len(object_references[1].map_squares)-1][0].y+(2.5 * constant.FONT_SIZE)

    # Drawing texts on screens
    for i in range(len(list_of_texts)):
        screen.blit(list_of_texts[i], (70, below_map_y+(1.5 * i * constant.FONT_SIZE)))

    # Draw Q matrix
    draw_matrix_text(font, screen, object_references)
    # Draw the labyrinth map square ids    

    
    
# Main function, the first function called in the program
def main():

    # Create q-learning and other necessary objects for the window
    agent = create_agent()
    labyrinth = create_labyrinth(agent)
    screen = create_game_window()
    monospace_font, fps_clock, game_step_timed_event = create_window_objects()
    
    # Game loop
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT: 
                running = False      
            # Update the game for each game step (currently each constant.GAME_STEP_DELAY)
            if event.type == game_step_timed_event:

                #q_learning_solver.decide_next_move()
                screen.fill(constant.COLOR_DICT['white'])
                #draw_text_on_screen(monospace_font, screen, (agent, labyrinth, q_learning_solver))
                labyrinth.draw(screen)

        # true if agent in is target/goal position
        #if(q_learning_solver.is_agent_on_goal()):
        #    q_learning_solver.randomize_agent_position()
                
        # Update game cycle based on a fps target
        fps_clock.tick(constant.FPS_TARGET)
        # Update and possibly repaint window
        pygame.display.update()

    
main()

pygame 2.1.2 (SDL 2.0.18, Python 3.8.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
test2=test

In [ ]:
selected_coordinates

(array([0, 5], dtype=int8), array([3, 4], dtype=int8))

In [ ]:
hostage_coord[0]

array([0, 5], dtype=int8)

In [ ]:
di, dj = hostage_coord[0]
print(di,dj)

0 5


In [ ]:
y=np.arange(test.shape[0] * test.shape[1]).reshape(test.shape[0], test.shape[1])
border_coord = list(np.unique(np.array(sorted(np.concatenate([y[0,:],y[-1,:],y[:,0], y[:,-1]]))).reshape(1,-1)))
selected_indices = np.random.choice( np.arange(len(border_coord)), 2 )
selected_coordinates = border_coord[0][selected_indices], border_coord[1][selected_indices]

selected_coordinates

TypeError: object of type 'numpy.int32' has no len()

In [ ]:
def discrete_int(n: int, start: int = 0):
        assert n > 0, "n (counts) have to be positive"
        assert isinstance(start, (int, np.integer))
        n = int(n)
        start = int(start)
        return ((), np.int64)

In [ ]:
discrete_int(2)

((), numpy.int64)

In [ ]:
num_states = 500
num_rows = 6
num_columns = 6
max_row = num_rows - 1
max_col = num_columns - 1


def testf(agent_row, agent_col, host_idx, drop_idx):

    i = agent_row
    i *= 6
    i += agent_col
    i *= 4
    i += host_idx
    i *= 2
    i += drop_idx
    return i

def testg(agent_row, agent_col, host_idx, exit_idx):

    i = agent_row
    i *= 6
    print(i)
    i += agent_col
    print(i)
    i *= 4
    print(i)
    i += host_idx
    print(i)
    i *= 2
    print(i)
    i += exit_idx
    print(i)
    return i

for row in range(num_rows):
    for col in range(num_columns):
        for host_idx in range(4):  # +2 for being with agent
            for exit_idx in range(2):
                state = testf(row, col, host_idx, exit_idx)
                if state == 287:
                    testg(row, col, host_idx, exit_idx)

30
35
140
143
286
287


In [ ]:
#(5, 5, 3, 1) # (taxi row, taxi column, passenger index, destination index)
testg(5, 5, 3, 1)

30
35
140
142
284
286


286

In [ ]:
np.random

<module 'numpy.random' from 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python38\\lib\\site-packages\\numpy\\random\\__init__.py'>

## Sources

In [ ]:
1)https://www.samyzaf.com/ML/rl/qmaze.html

2)
https://www.learndatasci.com/tutorials/reinforcement-q-learning-scratch-python-openai-gym/
https://github.com/openai/gym/blob/master/gym/envs/toy_text/taxi.py
    
3)https://github.com/ronanmmurphy/Q-Learning-Algorithm/blob/main/Q-Learning%20Algorithm.py

# Main class

In [326]:
action_dict = {
    0: 'up',
    1: 'down',
    2: 'right',
    3: 'left',
    4: 'pickup',
    5: 'dropoff',
    6: 'open'
}


color2num = dict(
    gray=30,
    red=31,
    green=32,
    yellow=33,
    blue=34,
    magenta=35,
    cyan=36,
    white=37,
    crimson=38,
)


def colorize(string, color, bold=False, highlight=False):
    """Return string surrounded by appropriate terminal color codes to
    print colorized text.  Valid colors: gray, red, green, yellow,
    blue, magenta, cyan, white, crimson
    """

    attr = []
    num = color2num[color]
    if highlight:
        num += 10
    attr.append(str(num))
    if bold:
        attr.append("1")
    attrs = ";".join(attr)
    return f"\x1b[{attrs}m{string}\x1b[0m"
def categorical_sample(prob_n, np_random):
    """
    Sample from categorical distribution
    Each row specifies class probabilities
    """
    prob_n = np.asarray(prob_n)
    csprob_n = np.cumsum(prob_n)
    return (csprob_n > np_random.random()).argmax()

class MapWalkingEnv:
    """

    ### Actions
    # There are 6 deterministic actions (discrete space):
      - 0: move down (south)
      - 1: move up (north)
      - 2: move left (west)
      - 3: move right (east)
      - 4: pickup hostage
      - 5: drop hostage
    
    
    ### States
    In total there are 500 discrete states.
    There are (grid size - number of walls) positions that can be ocupied by the agent.
    Considering a grid size of 6 x 6 and a total number of 10 walls there are 26 positions that can be ocupied by the agent,
    2 possible locations of the hostage zone (including the case when the passenger is in the
    taxi), and 2 locations of safe zone.
    
    """

    def __init__(self, N, M):
        # Numpy array that holds the information about the environment
        self.territory = np.zeros((N, M), dtype = np.int8)
        self.height = N
        self.width = M
        self.size = N * M 
        self.average_size = round((N+M)/2,0)
        self.hostage = 1 # 1 hostage zones
        self.exit = 2 # 1 safe zone and 1 unsafe
        
        # Number of random objects
        self.number_keys = 1
        self.number_hostages = 1
        self.number_mines = 1
        self.exit_steps = None 

        # Except 2 cells all borders are obstacles
        #self.territory[0, :] = 1
        #self.territory[-1, :] = 1
        #self.territory[:, 0] = 1
        #self.territory[:, -1] = 1
        
        #Pick N+M random coordinates for the obstacles
        #self.wall_coord = wall_coord = self.get_empty_cells( int(N+M) )
        #self.territory[wall_coord[0], wall_coord[1]] = 1
        
        wall_states = [(0,1),(1,5),(2,5),(3,0),(3,1),(3,3),(3,4),(3,6),(4,2),(4,4),(4,6),(4,7),(4,8),
                       (5,2),(5,4),(7,6),(7,7),(7,8),(7,9),(8,1),(8,2),(8,3),(8,4),(8,5),(9,7)]
        
        #Debugging purposes (smaller size)
        # wall_states = [(0,1),(1,5),(2,5),(3,3),(3,4),(3,6),(4,2),(4,4),(4,6),
        #                (5,2),(5,4)]
        
        for i,j in wall_states:
            self.territory[i,j] = 1
        
        # Pick 1 random coordinate for the agent location
        #empty_coord = self.get_empty_cells(1)
        #self.current_location = current_location = (empty_coord[0],empty_coord[1])
        
        # Pick 1 random coordinates for the mine
        #self.mine_coord = mine_coord = self.get_empty_cells(self.number_mines)
        self.mine_coord = mine_coord = (0,9)
        self.territory[mine_coord[0], mine_coord[1]] = 2

        # Pick 2 random coordinates over the border to be safe zone and unsafe zone
        self.exit_coord = exit_coord = self.get_exit_cells(self.exit)
        self.exit_coord = exit_coord = ((0,8),(9,9))  #Debugging purposes (initial seed)
        self.territory[exit_coord[0][0], exit_coord[0][1]] = 5 # Safe
        self.territory[exit_coord[1][0], exit_coord[1][1]] = 6 # Unsafe
        
        # Pick 1 random coordinate for the key
        self.key_coord = key_coord = self.get_empty_cells(self.number_keys)
        self.key_coord = key_coord = (5,7) #Debugging purposes (initial seed)
        self.territory[key_coord[0],key_coord[1]] = 3
         
        # Pick 1 random coordinate for the hostage zone
        self.hostage_coord = hostage_coord = self.get_empty_cells(self.number_hostages)
        self.hostage_coord = hostage_coord = (9,6) #(2,4) #Debugging purposes (initial seed)
        self.territory[hostage_coord[0],hostage_coord[1]] = 4
        
        
        
        # initial position of the agent will be decided by resetting the environment.
        self.position_agent = None
        self.start_agent = None

        # run time
        self.time_elapsed = 0
        self.time_limit = 2*self.size #* 8

        # display help
        self.dict_map_display={ 0:'.', #0 : empty cell
                                1:'X', #1 : walls obstacle that are non-traversable
                                2:'M', #2 : mine field location
                                3:'K', #3 : key location
                                4:'H', #3 : hostage zone    
                                5:'S', #4 : safe zone
                                6:'U', #6 : unsafe zone
                                7:'A' #7 : agent location
                              }
       

        num_states = self.size * 8
        num_rows = N
        num_columns = M
        max_row = num_rows - 1
        max_col = num_columns - 1
        self.initial_state_distrib = np.zeros(num_states)
        num_actions = 7
        
        # Set available actions
        self.actions = [0,1,2,3,4,5,6]
        #self.actions = ['up','down','left','right','pickup','dropoff','open']
        
        self.P = {
            state: {action: [] for action in range(num_actions)}
            for state in range(num_states)
        }
        self.states_array = {
            state   : {}
            for state in range(num_states)
        }
        self.observations_array = {
            state   : {}
            for state in range(num_states)
        }
        
        # Create matrix of transiction probabilities
        for row in range(num_rows): #7
            for col in range(num_columns): #7
                for key_idx in range(2): # +1 for key being with agent
                    for host_idx in range(2): # +1 for host being with agent
                        for exit_idx in range(2): # for tracking if door was opened or not
                            state = self.encode(row, col, key_idx, host_idx, exit_idx)
                            #if state == 69:
                            #    print(row, col, key_idx, host_idx, exit_idx)
                             
                            if  (self.territory[row, col] != 1) \
                                and self.territory[row, col] != 2 \
                                and (key_idx < 1 and host_idx < 1 and exit_idx < 1):
                                #and ((row, col) != (key_coord[0],key_coord[1])) \
                                #and ((row, col) != (hostage_coord[0],hostage_coord[1])) \
                                #key_idx < 1 and host_idx < 1 and \
                                self.initial_state_distrib[state] += 1
                                self.states_array[state] = (row, col)
                            if  self.territory[row, col] != 1: 
                                self.observations_array[state] = (row, col)
                            
                            valid_actions = self.valid_actions(row, col)
                            
                            for action in valid_actions:
                                # defaults
                                new_row, new_col, new_key_idx, new_host_idx, new_exit_idx=row, col, key_idx, host_idx, exit_idx
                                reward = (
                                    -0.1
                                )  # Default reward when there is no pickup/dropoff/open and agent moves to an open cell
                                
                                done = False
                                agent_loc = (row, col)
                                

                                if action == 0: #'up' 
                                    new_row = max(row - 1, 0)
                                elif action == 1: #'down' 
                                    new_row = min(row + 1, max_row)
                                elif action == 2: #'right' 
                                    new_col = min(col + 1, max_col)
                                elif action == 3: #'left':
                                    new_col = max(col - 1, 0)
                                elif action == 4: #'pickup' 
                                    if new_host_idx == 0 and new_key_idx == 1 and new_exit_idx == 1 and \
                                    (agent_loc == (hostage_coord[0],hostage_coord[1])):
                                        new_host_idx = 1
                                        reward = 20
                                    elif new_key_idx == 0 and (agent_loc == (key_coord[0],key_coord[1])):
                                        new_key_idx = 1
                                        reward = 20
                                    elif self.territory[new_row, new_col] != 2: # key not at location or agent did not open a door yet
                                        reward = -20
                                elif action == 5: #'dropoff' 
                                    if (agent_loc == (exit_coord[0][0],exit_coord[0][1]) and \
                                        new_key_idx == 1 and new_host_idx == 1):
                                        done = True
                                        reward = 100
                                    elif agent_loc == (exit_coord[1][0],exit_coord[1][1]) : #(agent_loc == (exit_coord[0][0],exit_coord[0][1]) and (new_key_idx == 0)) or \
                                        done = True
                                        reward = -100
                                    elif self.territory[new_row, new_col] != 2:  # dropoff at wrong location
                                        reward = -20
                                elif action == 6: #'open' 
                                    if (agent_loc == (exit_coord[0][0],exit_coord[0][1])) and (new_key_idx == 1 and new_exit_idx == 0):
                                        new_exit_idx = 1
                                        reward = 20
                                    elif self.territory[new_row, new_col] != 2:  # open at wrong location
                                        reward = -20

                                if (new_host_idx == 1 and new_key_idx == 1) or \
                                    new_exit_idx == 1:
                                    new_exit_idx=1
                                else:
                                    new_exit_idx=0
                                    
                                # If the agent bumps into a wall, it doesn't move
                                if self.territory[new_row, new_col] == 1:
                                    new_row = row
                                    new_col = col
                                    reward = -1
                                
                                # If the agent bumps into a mine it reaches a terminal state:
                                if self.territory[new_row, new_col] == 2:
                                    new_row = row
                                    new_col = col
                                    done = True
                                    reward = -100 

                                if self.territory[row, col] != 1 and self.territory[row, col] != 2:
                                    new_state = self.encode(
                                        new_row, new_col, new_key_idx, new_host_idx, new_exit_idx
                                    )
    #                                 if ((row, col) != (exit_coord[0][0],exit_coord[0][1])) \
    #                                     and ((row, col) != (exit_coord[1][0],exit_coord[1][1])):
                                    self.P[state][action].append((1.0, new_state, reward, done))
                       
        self.initial_state_distrib /= self.initial_state_distrib.sum()
        self.action_space = int(num_actions)
        self.state_space = int(num_states)
    
    def get_empty_cells(self, n_cells):
        empty_cells_coord = np.where( self.territory == 0 )
        selected_indices = np.random.choice( np.arange(len(empty_cells_coord[0])), n_cells )
        selected_coordinates = empty_cells_coord[0][selected_indices], empty_cells_coord[1][selected_indices]
        
        if n_cells == 1:
            return np.asarray(selected_coordinates).reshape(2,)
        
        return selected_coordinates
        
    def get_exit_cells(self, n_cells):
        y=np.arange(self.territory.shape[0] * self.territory.shape[1]).reshape(self.territory.shape[0], self.territory.shape[1])        
        border_coord_indices = list(np.unique(np.array(sorted(np.concatenate([y[0,:],y[-1,:],y[:,0], y[:,-1]]))).reshape(1,-1)))
        border_coord=np.zeros((2,len(border_coord_indices)),dtype = np.int8)
        border_coord[0] = [int(i) for i,ival in enumerate(y) for j,jval in enumerate(ival) if y[i,j] in border_coord_indices]
        border_coord[1] = [j for i,ival in enumerate(y) for j,jval in enumerate(ival) if y[i,j] in border_coord_indices]
        #print(border_coord_indices)
        #print(border_coord)
        selected_indices = np.random.choice( np.arange(len(border_coord[0])), n_cells )
        selected_coordinates = border_coord[0][selected_indices], border_coord[1][selected_indices]
        
        if n_cells == 1:
            return np.asarray(selected_coordinates).reshape(2,)
        
        return selected_coordinates
    
    

    def step(self, action):
        
        # At every timestep, the agent receives a negative reward
        
        
#         key_idx < 1 and host_idx < 1 and     
#         and ((row, col) != (exit_coord[0][0],exit_coord[0][1])) \
#         and ((row, col) != (exit_coord[1][0],exit_coord[1][1]))
        
        
        transitions = self.P[self.s][action]
        i = categorical_sample([t[0] for t in transitions], np.random)
        p, state, reward, done = transitions[i]
        self.s = state
        self.lastaction = action
        #print(self.observations_array[self.s])
        
        if done == False:
            self.position_agent = (self.observations_array[self.s][0],self.observations_array[self.s][1])

        # Update time elapsed
        self.time_elapsed+=1
        
        _, _, _, _, self.exit_steps = env.decode(env.s)

        if self.time_elapsed == self.time_limit:
            done = True
        
        #if (self.position_agent == self.position_exit).all():
        #    done = True
        return (int(state), reward, done, {"prob": p})
   
    
    def agent_on_map(self):
        """Prints out current location of the agent on the grid (used for debugging)"""
        grid = np.zeros((self.N, self.M))
        grid[ self.current_location[0], self.current_location[1]] = 1
        return grid
    
    def encode(self, agent_row, agent_col, key_idx, host_idx, exit_idx):
        i = agent_row
        i *= 10
        i += agent_col
        i *= 2
        i += key_idx
        i *= 2
        i += host_idx
        i *= 2
        i += exit_idx
        return i

    def decode(self, i):
        out = []
        out.append(i % 2)
        i = i // 2
        out.append(i % 2)
        i = i // 2
        out.append(i % 2)
        i = i // 2
        out.append(i % 10)
        i = i // 10
        out.append(i)
        assert 0 <= i < 10
        return reversed(out)
    
    def display(self, mode='human'):
 

        
        envir_with_agent = self.territory.copy()
        envir_with_agent[self.position_agent[0], self.position_agent[1]] = 7
        #envir_with_agent[self.position_agent[0], self.position_agent[1]] = 7
        
        agent_row, agent_col, key_idx, host_idx, exit_idx = self.decode(self.s)

        full_repr = ""

        for r in range(self.height):
            
            line = ""
            
            for c in range(self.width):
                #if (key_idx == 0 or host_idx == 0):
                #             out[1 + agent_row][2 * agent_col + 1] = utils.colorize(
                #                 out[1 + agent_row][2 * agent_col + 1], "yellow", highlight=True
                #             )
                #teststr = colorize('A', "green", highlight=True)
                
                string_repr = self.dict_map_display[ envir_with_agent[r,c] ]
                
                    #print(colorize('A', "green", highlight=True))
                line += "{0:2}".format(string_repr)

            full_repr += line + "\n"
        
        if key_idx == 1 and exit_idx == 0: # Agent found the key but did not open the door yet
            full_repr = full_repr.replace('A',colorize('A', "yellow", highlight=True))
        elif exit_idx == 1 and host_idx == 0: # Agent opened the door but did not find hostage
            full_repr = full_repr.replace('A',colorize('A', "blue", highlight=True))
        elif host_idx == 1: # Agent found the hostage
            full_repr = full_repr.replace('A',colorize('A', "green", highlight=True))

        print('Key (Yellow):',key_idx,'/ Door (Blue):',exit_idx,'/ Hostage (Green):',host_idx,'- Agent start',self.start_agent)
        print(full_repr)
        

         #if pass_idx < 4
#         if (key_idx == 0 or host_idx == 0):
#             out[1 + agent_row][2 * agent_col + 1] = utils.colorize(
#                 out[1 + agent_row][2 * agent_col + 1], "yellow", highlight=True
#             )
#             #pi, pj = self.locs[pass_idx]
#             pi, pj = (key_coord[0],key_coord[1]) #self.key_idx
#             out[1 + pi][2 * pj + 1] = utils.colorize(
#                 out[1 + pi][2 * pj + 1], "blue", bold=True
#             )
#         else:  # passenger in taxi
#             out[1 + agent_row][2 * agent_col + 1] = utils.colorize(
#                 ul(out[1 + agent_row][2 * agent_col + 1]), "green", highlight=True
#             )
        
#         di, dj = exit_coord[0] #self.locs[exit_idx]
#         out[1 + di][2 * dj + 1] = utils.colorize(out[1 + di][2 * dj + 1], "magenta")
#         outfile.write("\n".join(["".join(row) for row in out]) + "\n")
#         if self.lastaction is not None:
#             outfile.write(
#                 f"  ({['Down', 'Up', 'Right', 'Left', 'Pickup', 'Dropoff'][self.lastaction]})\n"
#             )
#         else:
#             outfile.write("\n")

#         # No need to return anything for human
#         if mode != "human":
#             with closing(outfile):
#                 return outfile.getvalue()
        
        

        
    def calculate_observations(self):
        
        relative_coordinates = self.position_exit - self.position_agent
                
        surroundings = self.territory[ self.position_agent[0] -1: self.position_agent[0] +2,
                                     self.position_agent[1] -1: self.position_agent[1] +2]
        
        obs = {'relative_coordinates':relative_coordinates,
               'surroundings': surroundings}
        
        return obs
        
    def reset(self, return_info: bool = False):
        """
        This function resets the environment to its original state (time = 0).
        Then it places the agent and exit at new random locations.
        
        It is common practice to return the observations, 
        so that the agent can decide on the first action right after the resetting of the environment.
        
        """
        self.time_elapsed = 0
         
        invalid_position = True                                    


        while invalid_position == True:
            self.s = categorical_sample(self.initial_state_distrib, np.random)
            new_coord = self.states_array[self.s]
            #print(new_coord)
            if      ((new_coord[0],new_coord[1]) != (self.exit_coord[0][0],self.exit_coord[0][1])) \
                and ((new_coord[0],new_coord[1]) != (self.exit_coord[1][0],self.exit_coord[1][1])) \
                and ((new_coord[0],new_coord[1]) != (self.hostage_coord[0],self.hostage_coord[1])) \
                and ((new_coord[0],new_coord[1]) != (self.key_coord[0],self.key_coord[1])):
                invalid_position = False
                self.position_agent = (new_coord[0],new_coord[1])
        
        #self.position_agent = (6,9)
        self.start_agent = self.position_agent


        # Find a position that is not close to an object
#         while True:
#             self.s = categorical_sample(self.initial_state_distrib, np.random)
#             new_coord = my_territory.states_array[self.s]
#             self.position_agent = (new_coord[0],new_coord[1])
#             surrounding_cells = self.territory[max(self.position_agent[0] -1,0): self.position_agent[0] +2,
#                                      max(self.position_agent[1] -1,0): self.position_agent[1] +2].sum()
#             if surrounding_cells == 0:
#                 break
        
        # Calculate observations
        #observations = self.calculate_observations()
        

        self.lastaction = None
        if not return_info:
            return int(self.s)
        else:
            return int(self.s), {"prob": 1}
        #return observations
        
        
    def valid_actions(self, row=None, col=None, cell=None):
        actions = [0, 1, 2, 3, 4, 5, 6]
        if (row == None and col == None):
            row = self.observations_array[self.s][0]
            col = self.observations_array[self.s][1]

        nrows, ncols = self.territory.shape
        if row == 0:
            actions.remove(0)
        elif row == nrows-1:
            actions.remove(1)

        if col == 0:
            actions.remove(3)
        elif col == ncols-1:
            actions.remove(2)

#         if row>0 and self.territory[row-1,col] == 1:
#             actions.remove(0)
#         if row<nrows-1 and self.territory[row+1,col] == 1:
#             actions.remove(1)

#         if col>0 and self.territory[row,col-1] == 1:
#             actions.remove(3)
#         if col<ncols-1 and self.territory[row,col+1] == 1:
#             actions.remove(2)
        return actions

In [ ]:
print("Current position of the agent =", env.current_location)
print(env.agent_on_map())
available_actions = env.get_available_actions()
print("Available_actions =", available_actions)
chosen_action = agent.choose_action(available_actions)
print("Randomly chosen action =", chosen_action)
reward = env.make_step(chosen_action)
print("Reward obtained =", reward)
print("Current position of the agent =", env.current_location)
print(env.agent_on_map())

500

In [205]:

_, _, _, _, exit_step = env.decode(env.s)
print(exit_step)

0


# Q-Learning Training

In [ ]:
%%time
"""Training the agent"""

import time 
import numpy as np
import random
from IPython.display import clear_output

# Variables
N = 10
M = 10
episodes = 50000

# Hyperparameters
alpha = 0.9 #0.1
gamma = 0.9 #0.6

# Exploration rate.
epsilon = 0.9 #0.1

# Exploration decay rate.
DECAY_RATE = 0.99997 #0.9998          

# For plotting metrics
all_epochs = []
all_penalties = []


env = MapWalkingEnv(N,M)

Q = np.zeros([env.state_space, env.action_space])

for i in range(1, episodes):
    state = env.reset()
    
    epochs, penalties, reward, = 0, 0, 0
    done = False
    
    exit_num_steps = 0

    while not done:
        valid_actions = env.valid_actions()
        if not valid_actions: break
        if random.uniform(0, 1) < epsilon:
            #action = env.action_space.sample() # Explore action space
            action = np.random.choice(valid_actions)
        else:
            #action = np.argmax(Q[state]) # Exploit learned values
            action_value = np.array([-np.inf,-np.inf,-np.inf,-np.inf,-np.inf,-np.inf,-np.inf])
            for action,value in enumerate(Q[state]):
                if action in valid_actions:
                    action_value[action] = value
            action = np.random.choice(np.flatnonzero(action_value == action_value.max())) # Exploit learned values
            #action = np.argmax(action_value) # Exploit learned values
            
        next_state, reward, done, info = env.step(action) 
        
        #print(action_dict[action])
        #print('Time Elapsed:',env.time_elapsed,'State:',env.s,'Action:',action_dict[action],'Reward:',reward)
        #env.display()

        old_value = Q[state, action]
        next_max = np.max(Q[next_state])
        
        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        Q[state, action] = new_value

        #if reward == -10:
        #    penalties += 1

        if env.exit_steps == 1:
            exit_num_steps  = epochs 

        state = next_state
        epochs += 1

        

    if epsilon > 0.15:
        epsilon *= DECAY_RATE                 # Decaying the exploration rate every episode.
    else:
        epsilon = 0.15
    # if alpha > 0.3:
    #     alpha *= DECAY_RATE                   # Decaying the learning rate every episode.
    # else:
    #     alpha = 0.3
    
    if i % 100 == 0:
        clear_output(wait=True)
        print(f"Episode: {i}")
        env.display()
        print('Time Elapsed:',env.time_elapsed,'-','State:',env.s,'-','Action:',action_dict[action],'-','Reward:',reward,
        print('Steps to open door:',exit_num_steps, '-','Steps to complete:',exit_num_steps))
        
print("Training finished.\n")

Episode: 11900
Key (Yellow): 1 / Door (Blue): 1 / Hostage (Green): 0 - Agent start (7, 4)
. X . . . . . . S M 
. . . . . X . . . A 
. . . . . X . . . . 
X X . X X . X . . . 
. . X . X . X X X . 
. . X . X . . K . . 
. . . . . . . . . . 
. . . . . . X X X X 
. X X X X X . . . . 
. . . . . . H X . U 

Steps to open door: 40 - Steps to complete: 40
Time Elapsed: 41 - State: 157 - Action: up - Reward: -100 None


In [325]:
0.9 * pow(0.99997,30000)

0.36590775387999974

## Evaluation

In [280]:
#action = np.argmax(Q[state]) # Exploit learned values
action_value = np.array([-np.inf,-np.inf,-np.inf,-np.inf,-np.inf,-np.inf,-np.inf])
for action,value in enumerate(Q[state]):
    if action in valid_actions:
        action_value[action] = value
action = np.random.choice(np.flatnonzero(action_value == action_value.max())) # Exploit learned values
#action = np.argmax(action_value) # Exploit learned values

next_state, reward, done, info = env.step(action) 
print(next_state, reward, done, info)

476 -0.01 False {'prob': 1.0}


In [289]:
agent_row, agent_col, key_idx, host_idx, exit_idx = env.decode(152)
print(agent_row, agent_col, key_idx, host_idx, exit_idx)

agent_row, agent_col, key_idx, host_idx, exit_idx = env.decode(156)
print(agent_row, agent_col, key_idx, host_idx, exit_idx)


1 9 0 0 0
1 9 1 0 0


In [316]:
"""Evaluate agent's performance after Q-learning"""

total_epochs, total_penalties = 0, 0
episodes = 2

for _ in range(episodes):
    state = env.reset()
    epochs, penalties, reward = 0, 0, 0
    
    done = False
    #env.display()
    while not done:
        valid_actions = env.valid_actions()
        if not valid_actions: break
        action_value = np.array([-np.inf,-np.inf,-np.inf,-np.inf,-np.inf,-np.inf,-np.inf])
        for action,value in enumerate(Q[state]):
            if action in valid_actions:
                action_value[action] = value
        action = np.random.choice(np.flatnonzero(action_value == action_value.max())) # Exploit learned values
        
        state, reward, done, info = env.step(action)

        if epochs < 50:
            print(env.time_elapsed,'-',env.s,'-',action_dict[action],'-',reward,'-',action)
            env.display()
        
        if reward < -1:
            penalties += 1

        
        epochs += 1

    total_penalties += penalties
    total_epochs += epochs

print(f"Results after {episodes} episodes:")
print(f"Average timesteps per episode: {total_epochs / episodes}")
print(f"Average penalties per episode: {total_penalties / episodes}")
print(f"Total penalties (Invalid pickups/dropoffs/open): {total_penalties}")

1 - 168 - right - -0.01 - 2
Key (Yellow): 0 / Door (Blue): 0 / Hostage (Green): 0 - Agent start (2, 0)
. X . . . . . . S M 
. . . . . X . . . . 
. A . . . X . . . . 
X X . X X . X . . . 
. . X . X . X X X . 
. . X . X . . K . . 
. . . . . . . . . . 
. . . . . . X X X X 
. X X X X X . . . . 
. . . . . . H X . U 

2 - 176 - right - -0.01 - 2
Key (Yellow): 0 / Door (Blue): 0 / Hostage (Green): 0 - Agent start (2, 0)
. X . . . . . . S M 
. . . . . X . . . . 
. . A . . X . . . . 
X X . X X . X . . . 
. . X . X . X X X . 
. . X . X . . K . . 
. . . . . . . . . . 
. . . . . . X X X X 
. X X X X X . . . . 
. . . . . . H X . U 

3 - 96 - up - -0.01 - 0
Key (Yellow): 0 / Door (Blue): 0 / Hostage (Green): 0 - Agent start (2, 0)
. X . . . . . . S M 
. . A . . X . . . . 
. . . . . X . . . . 
X X . X X . X . . . 
. . X . X . X X X . 
. . X . X . . K . . 
. . . . . . . . . . 
. . . . . . X X X X 
. X X X X X . . . . 
. . . . . . H X . U 

4 - 104 - right - -0.01 - 2
Key (Yellow): 0 / Door (Blue): 0 /